# AgentPoison Paper Recreation Notebook

This notebook is the working control panel for recreating **AgentPoison: Red-teaming LLM Agents via Memory or Knowledge Base Backdoor Poisoning** on **Google Colab**.

Primary target for the proposal:

- Start with **ReAct-StrategyQA (`qa`)** and **EHRAgent (`ehr`)**.
- Reproduce Table 1 cells for AgentPoison first, then add baselines if time permits.
- Reproduce Figure 4 sweeps for poison count and trigger length.
- Reproduce Table 2 loss ablations.
- Reproduce simple defenses: perplexity filtering and query rephrasing.
- Save every plot/table artifact into `outputs/figures` or `outputs/tables` for slides.

The setup cells below clone the official repo into `/content` or Google Drive and configure output folders for Colab.

## Run Order

This version is Colab-first.

1. Select a GPU runtime in Colab: `Runtime > Change runtime type > GPU`.
2. Run the Colab setup cells to mount Drive if desired, clone AgentPoison, and install dependencies.
3. Upload/download datasets into the expected repo folders.
4. Verify GPU, repo, and dataset layout.
5. Optimize triggers for `qa` and `ehr`.
6. Run each agent twice: attacked input for ASR, benign input for ACC.
7. Run official eval scripts and collect metrics.
8. Generate tables and figures for the writeup/presentation.

`RUN_JOBS` defaults to `False` so the notebook prints commands first. Flip it to `True` only after Colab setup and preflight checks pass.

## Colab Setup

Run these cells at the top of a fresh Colab session. For long runs, storing outputs in Google Drive is strongly recommended because `/content` is deleted when the runtime disconnects.

Set `USE_GOOGLE_DRIVE = True` if you want persistent outputs under `MyDrive/agentpoison_reproduction`. If you keep it `False`, everything stays under `/content/agentpoison_reproduction`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
USE_GOOGLE_DRIVE = True

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/agentpoison_reproduction")
elif IN_COLAB:
    PROJECT_ROOT = Path("/content/agentpoison_reproduction")
else:
    PROJECT_ROOT = Path.cwd()

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
AGENTPOISON_ROOT = PROJECT_ROOT / "AgentPoison"

print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("AGENTPOISON_ROOT:", AGENTPOISON_ROOT)

In [ ]:
# Clone or update the official repository.
# Use BillChan226/AgentPoison because that is the URL in the paper README.
if not AGENTPOISON_ROOT.exists():
    subprocess.run([
        "git", "clone", "https://github.com/BillChan226/AgentPoison.git", str(AGENTPOISON_ROOT)
    ], check=True)
else:
    print("Repo already exists:", AGENTPOISON_ROOT)

print("Repo contents preview:")
print("\n".join(sorted(p.name for p in AGENTPOISON_ROOT.iterdir())[:20]))

In [ ]:
# Colab compatibility patch for newer transformers.
# The AgentPoison repo imports RealmEmbedder, which was removed from recent transformers exports.
# Our first reproduction uses DPR, not REALM, so this fallback lets DPR runs import cleanly.
from pathlib import Path

compat_replacements = {
    AGENTPOISON_ROOT / "algo" / "trigger_optimization.py": {
        "root_dir = f\"{args.save_dir}/{config.agent}/{config.algo}/{str(datetime.datetime.now())}\"": "root_dir = f\"{args.save_dir}/{args.agent}/{args.algo}/{str(datetime.datetime.now())}\"",
    },
    AGENTPOISON_ROOT / "algo" / "utils.py": {
        "from transformers import (BertModel, \n                          BertTokenizer, \n                          AutoModelForCausalLM, \n                          LlamaForCausalLM, \n                          DPRContextEncoder,\n                          AutoModel,\n                          DPRQuestionEncoder,\n                          RealmEmbedder,\n                          RealmForOpenQA)": "from transformers import (BertModel,\n                          BertTokenizer,\n                          AutoModelForCausalLM,\n                          LlamaForCausalLM,\n                          DPRContextEncoder,\n                          AutoModel,\n                          DPRQuestionEncoder)\ntry:\n    from transformers import RealmEmbedder, RealmForOpenQA\nexcept ImportError:\n    class RealmEmbedder:  # Colab/new-transformers fallback; REALM runs are disabled.\n        @classmethod\n        def from_pretrained(cls, *args, **kwargs):\n            raise ImportError('RealmEmbedder is unavailable in this transformers version. Use dpr/ance/bge or install an older transformers environment.')\n\n    class RealmForOpenQA:\n        @classmethod\n        def from_pretrained(cls, *args, **kwargs):\n            raise ImportError('RealmForOpenQA is unavailable in this transformers version. Use dpr/ance/bge or install an older transformers environment.')",
    },
    AGENTPOISON_ROOT / "ReAct" / "local_wikienv.py": {
        "from transformers import AutoTokenizer, DPRContextEncoder, RealmEmbedder": "from transformers import AutoTokenizer, DPRContextEncoder\ntry:\n    from transformers import RealmEmbedder\nexcept ImportError:\n    class RealmEmbedder:  # Colab/new-transformers fallback; REALM runs are disabled.\n        @classmethod\n        def from_pretrained(cls, *args, **kwargs):\n            raise ImportError('RealmEmbedder is unavailable in this transformers version. Use dpr/ance/bge or install an older transformers environment.')",
    },
    AGENTPOISON_ROOT / "agentdriver" / "memory" / "memory_agent.py": {
        "from transformers import (BertModel, BertTokenizer, DPRContextEncoder, AutoModel, RealmEmbedder, RealmForOpenQA)": "from transformers import BertModel, BertTokenizer, DPRContextEncoder, AutoModel\ntry:\n    from transformers import RealmEmbedder, RealmForOpenQA\nexcept ImportError:\n    class RealmEmbedder:  # Colab/new-transformers fallback; REALM runs are disabled.\n        @classmethod\n        def from_pretrained(cls, *args, **kwargs):\n            raise ImportError('RealmEmbedder is unavailable in this transformers version. Use dpr/ance/bge or install an older transformers environment.')\n\n    class RealmForOpenQA:\n        @classmethod\n        def from_pretrained(cls, *args, **kwargs):\n            raise ImportError('RealmForOpenQA is unavailable in this transformers version. Use dpr/ance/bge or install an older transformers environment.')",
    },
}

for file_path, replacements in compat_replacements.items():
    if not file_path.exists():
        print("Skipping missing file:", file_path)
        continue
    text = file_path.read_text()
    changed = False
    for old, new in replacements.items():
        if old in text:
            text = text.replace(old, new)
            changed = True
    if changed:
        file_path.write_text(text)
        print("Patched:", file_path)
    else:
        print("Already patched or pattern not found:", file_path)

In [ ]:
# Colab dependency install.
# Colab uses Python 3.12 and a preinstalled CUDA/PyTorch stack, so avoid the repo's old pins.
# This installs the packages needed for qa/ehr first, then tries stretch/optional packages separately.
INSTALL_DEPS = True

CORE_DEPS = [
    "transformers",
    "sentence-transformers",
    "accelerate",
    "datasets",
    "scikit-learn",
    "seaborn",
    "omegaconf",
    "iopath",
    "jsonlines",
    "shortuuid",
    "termcolor",
    "wandb",
    "openai",
    "peft",
    "timm",
    "webdataset",
    "beautifulsoup4",
    "gym",
    "python-Levenshtein",
    "wolframalpha",
    "replicate",
]

OPTIONAL_DEPS = [
    # Mostly needed for Agent-Driver/stretch experiments. Do not block qa/ehr if one fails.
    "decord",
    "bitsandbytes",
    "casadi",
    "opencv-python",
]


def pip_install(packages, required=True):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print("Installing:", " ".join(packages))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        message = f"pip install failed for: {packages}"
        if required:
            raise RuntimeError(message)
        print("Optional dependency skipped:", message)
    return result.returncode == 0


if INSTALL_DEPS:
    pip_install(CORE_DEPS, required=True)
    for package in OPTIONAL_DEPS:
        pip_install([package], required=False)
else:
    print("INSTALL_DEPS is False. Set it to True in Colab if imports fail.")

# Quick sanity check. If this prints a GPU name, Colab sees the selected accelerator.
subprocess.run([
    sys.executable,
    "-c",
    "import torch, transformers, pandas, numpy, matplotlib; "
    "print('torch', torch.__version__); "
    "print('transformers', transformers.__version__); "
    "print('cuda', torch.cuda.is_available()); "
    "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')",
], check=False)

In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# PROJECT_ROOT and AGENTPOISON_ROOT are set by the Colab setup cell above.
# If you skipped that cell outside Colab, fall back to the current directory.
PROJECT_ROOT = globals().get("PROJECT_ROOT", Path.cwd())
AGENTPOISON_ROOT = globals().get("AGENTPOISON_ROOT", PROJECT_ROOT / "AgentPoison")
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
FIG_DIR = OUTPUT_ROOT / "figures"
TABLE_DIR = OUTPUT_ROOT / "tables"
LOG_DIR = OUTPUT_ROOT / "logs"
RESULT_DIR = OUTPUT_ROOT / "agentpoison_runs"

for path in [OUTPUT_ROOT, FIG_DIR, TABLE_DIR, LOG_DIR, RESULT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Keep this False until Colab setup, dataset placement, and GPU checks pass.
RUN_JOBS = False

# Colab exposes the selected GPU as cuda:0. For multi-GPU runtimes, change this.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# Fill these in only in your private runtime, not in git/notebook submissions.
os.environ.setdefault("OPENAI_API_KEY", "")
os.environ.setdefault("WANDB_DISABLED", "true")

print("Project root:", PROJECT_ROOT)
print("AgentPoison root:", AGENTPOISON_ROOT)
print("Outputs:", OUTPUT_ROOT)

In [ ]:
def run_command(cmd: list[str] | str, cwd: Path = AGENTPOISON_ROOT, run: bool = RUN_JOBS) -> subprocess.CompletedProcess | None:
    """Print or run a shell command, saving stdout/stderr logs for long experiments."""
    if isinstance(cmd, list):
        printable = " ".join(shlex.quote(str(part)) for part in cmd)
    else:
        printable = cmd

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / f"{timestamp}.log"
    print(f"[cwd={cwd}] {printable}")
    print(f"log: {log_path}")

    if not run:
        return None

    proc = subprocess.run(
        cmd,
        cwd=cwd,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )
    log_path.write_text(proc.stdout)
    print(proc.stdout[-4000:])
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}. See {log_path}")
    return proc


def savefig(name: str, fig=None, dpi: int = 200) -> tuple[Path, Path]:
    """Save slide-ready PNG and paper-ready PDF versions of the current figure."""
    fig = fig or plt.gcf()
    png_path = FIG_DIR / f"{name}.png"
    pdf_path = FIG_DIR / f"{name}.pdf"
    fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print("saved", png_path)
    print("saved", pdf_path)
    return png_path, pdf_path


def write_table(df: pd.DataFrame, name: str) -> Path:
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print("saved", path)
    return path

In [ ]:
# Hardware and repo preflight. Run this after selecting a Colab GPU runtime.
run_command("pwd && python --version && nvidia-smi", cwd=PROJECT_ROOT, run=RUN_JOBS)

required_paths = [
    AGENTPOISON_ROOT / "algo" / "trigger_optimization.py",
    AGENTPOISON_ROOT / "ReAct",
    AGENTPOISON_ROOT / "EhrAgent",
    AGENTPOISON_ROOT / "environment.yml",
]

for path in required_paths:
    print(f"{path}: {'OK' if path.exists() else 'MISSING'}")

# Expected dataset placement from the official README.
dataset_paths = {
    "strategyqa": AGENTPOISON_ROOT / "ReAct" / "database",
    "ehr": AGENTPOISON_ROOT / "EhrAgent" / "database",
    "agent_driver_stretch": AGENTPOISON_ROOT / "agentdriver" / "data",
}
for name, path in dataset_paths.items():
    print(f"{name}: {path} -> {'OK' if path.exists() else 'MISSING'}")

print("\nColab data tip: upload datasets into Google Drive first, then copy/symlink them into the folders above.")

## Colab Environment Notes

Colab does not usually use the repo's `conda env create -f environment.yml` workflow. The setup cell above installs the non-PyTorch dependencies with `pip` and keeps Colab's preinstalled CUDA/PyTorch build.

If imports fail, set `INSTALL_DEPS = True` and rerun the dependency cell. If Colab gives you an A100/L4/T4 instead of an H100, the notebook still works, but trigger optimization will be slower. Start with one `qa` run before launching the full grid.

In [ ]:
@dataclass(frozen=True)
class TriggerJob:
    agent: str
    algo: str = "ap"
    model: str = "dpr-ctx_encoder-single-nq-base"
    trigger_len: int = 10
    num_iter: int = 1000
    num_grad_iter: int = 30
    num_cand: int = 100
    batch_size: int = 64
    seed: int = 0
    ppl_filter: bool = True
    # Colab smoke runs keep this off because --target_gradient_guidance loads
    # gated Llama-2 from a hard-coded author-local cache path.
    target_gradient_guidance: bool = False

    @property
    def save_dir(self) -> Path:
        return RESULT_DIR / "triggers" / self.agent / self.algo / self.model / f"len{self.trigger_len}_seed{self.seed}"

    def command(self) -> list[str]:
        cmd = [
            "python", "algo/trigger_optimization.py",
            "--agent", self.agent,
            "--algo", self.algo,
            "--model", self.model,
            "--save_dir", str(self.save_dir),
            "--num_iter", str(self.num_iter),
            "--num_grad_iter", str(self.num_grad_iter),
            "--per_gpu_eval_batch_size", str(self.batch_size),
            "--num_cand", str(self.num_cand),
            "--num_adv_passage_tokens", str(self.trigger_len),
            "--asr_threshold", "0.5",
            "--plot",
        ]
        if self.ppl_filter:
            cmd.append("--ppl_filter")
        if self.target_gradient_guidance:
            cmd.append("--target_gradient_guidance")
        return cmd


# Current checkout does not expose --seed in trigger_optimization.py.
# The seed field labels output folders until we add a real seed flag to the repo.
trigger_jobs = [TriggerJob(agent="qa", seed=s) for s in [0, 1, 2]] + [TriggerJob(agent="ehr", seed=s) for s in [0, 1, 2]]

pd.DataFrame(asdict(job) | {"save_dir": str(job.save_dir), "command": " ".join(job.command())} for job in trigger_jobs)

In [ ]:
# Print or run trigger optimization commands.
for job in trigger_jobs:
    job.save_dir.mkdir(parents=True, exist_ok=True)
    run_command(job.command())

In [ ]:
@dataclass(frozen=True)
class AgentRun:
    agent: str
    model: str = "dpr"
    algo: str = "ap"
    task_type: str = "adv"  # ReAct uses adv/benign.
    backbone: str = "gpt"   # EHRAgent uses gpt/llama3.
    seed: int = 0

    @property
    def save_dir(self) -> Path:
        return RESULT_DIR / "agent_runs" / self.agent / self.model / self.task_type / f"seed{self.seed}"

    def command(self) -> list[str]:
        if self.agent == "qa":
            return [
                "python", "ReAct/run_strategyqa_gpt3.5.py",
                "--model", self.model,
                "--task_type", self.task_type,
                "--save_dir", str(self.save_dir),
            ]
        if self.agent == "ehr":
            cmd = [
                "python", "EhrAgent/ehragent/main.py",
                "--backbone", self.backbone,
                "--model", self.model,
                "--algo", self.algo,
                "--save_dir", str(self.save_dir),
            ]
            if self.task_type == "adv":
                cmd.append("--attack")
            return cmd
        raise ValueError(f"Unsupported agent: {self.agent}")


agent_runs = []
for seed in [0, 1, 2]:
    agent_runs.extend([
        AgentRun(agent="qa", task_type="adv", seed=seed),
        AgentRun(agent="qa", task_type="benign", seed=seed),
        AgentRun(agent="ehr", task_type="adv", seed=seed),
        AgentRun(agent="ehr", task_type="benign", seed=seed),
    ])

pd.DataFrame({"agent": r.agent, "task_type": r.task_type, "save_dir": str(r.save_dir), "cmd": " ".join(r.command())} for r in agent_runs)

In [ ]:
# Print or run attacked and benign agent inference.
# Important: paste the optimized trigger into the repo locations described by the README before running.
for run in agent_runs:
    run.save_dir.mkdir(parents=True, exist_ok=True)
    run_command(run.command())

In [ ]:
def find_json_outputs(root: Path) -> list[Path]:
    return sorted(root.rglob("*.json")) + sorted(root.rglob("*.jsonl"))


def eval_command(agent: str, response_path: Path) -> list[str]:
    if agent == "qa":
        return ["python", "ReAct/eval.py", "-p", str(response_path)]
    if agent == "ehr":
        return ["python", "EhrAgent/ehragent/eval.py", "-p", str(response_path)]
    raise ValueError(agent)


# After inference finishes, this discovers response files and prints eval commands.
for run in agent_runs:
    outputs = find_json_outputs(run.save_dir)
    print("\n", run.agent, run.task_type, run.save_dir)
    if not outputs:
        print("No response JSON found yet.")
        continue
    for response_path in outputs[:5]:
        run_command(eval_command(run.agent, response_path))

## Result Registry

The official scripts print some metrics directly and save others in JSON/log files. Use this registry as the single source of truth for the paper, slides, and Medium post. Fill in `paper_value` from the paper and `ours_*` from each completed run, then rerun the plotting cells.

In [ ]:
metrics = ["ASR-r", "ASR-a", "ASR-t", "ACC"]
table1_rows = []
for agent in ["qa", "ehr"]:
    for method in ["AgentPoison", "GCG", "AutoDAN", "CPA", "BadChain"]:
        for metric in metrics:
            table1_rows.append({
                "figure_or_table": "Table 1",
                "agent": agent,
                "method": method,
                "model": "GPT-4o-mini substitution",
                "retriever": "DPR/contrastive",
                "metric": metric,
                "paper_value": np.nan,
                "ours_seed0": np.nan,
                "ours_seed1": np.nan,
                "ours_seed2": np.nan,
            })

table1 = pd.DataFrame(table1_rows)
table1["ours_mean"] = table1[["ours_seed0", "ours_seed1", "ours_seed2"]].mean(axis=1)
table1["ours_std"] = table1[["ours_seed0", "ours_seed1", "ours_seed2"]].std(axis=1)
table1["delta"] = table1["ours_mean"] - table1["paper_value"]
write_table(table1, "table1_registry")
table1.head(12)

In [ ]:
def plot_table1_summary(df: pd.DataFrame) -> None:
    ready = df[df["ours_mean"].notna()]
    if ready.empty:
        print("No completed metrics yet. Fill in the registry or add a parser for eval logs first.")
        return
    pivot = ready.pivot_table(index=["agent", "method"], columns="metric", values="ours_mean")
    ax = pivot.plot(kind="bar", figsize=(12, 5), rot=45)
    ax.set_ylabel("Score")
    ax.set_title("AgentPoison reproduction: Table 1 metrics")
    ax.legend(loc="best", ncols=4)
    plt.tight_layout()
    savefig("table1_reproduction_summary")


plot_table1_summary(table1)

In [ ]:
# Figure 4 sweeps: poison count and trigger length.
sweep_jobs = []
for agent in ["qa", "ehr"]:
    for trigger_len in [2, 4, 6, 8, 10]:
        sweep_jobs.append(TriggerJob(agent=agent, trigger_len=trigger_len, seed=0))

sweep_plan = pd.DataFrame(asdict(job) | {"save_dir": str(job.save_dir), "cmd": " ".join(job.command())} for job in sweep_jobs)
write_table(sweep_plan, "figure4_trigger_length_sweep_plan")
sweep_plan.head()

In [ ]:
# Run the trigger-length sweep after the base reproduction works.
for job in sweep_jobs:
    job.save_dir.mkdir(parents=True, exist_ok=True)
    run_command(job.command())

# For poison-count sweeps, edit the repo/config option that controls inserted poison entries if exposed by your local checkout.
# Record each run in outputs/tables/figure4_poison_count_results.csv using columns:
# agent, poison_count, trigger_len, seed, ASR-r, ASR-a, ASR-t, ACC

In [ ]:
# Plot Figure 4 once sweep results are filled in.
figure4_path = TABLE_DIR / "figure4_sweep_results.csv"
if figure4_path.exists():
    fig4 = pd.read_csv(figure4_path)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for agent, group in fig4.groupby("agent"):
        if "trigger_len" in group:
            axes[0].plot(group["trigger_len"], group["ASR-t"], marker="o", label=agent)
        if "poison_count" in group:
            axes[1].plot(group["poison_count"], group["ASR-t"], marker="o", label=agent)
    axes[0].set_title("ASR-t vs trigger length")
    axes[0].set_xlabel("Trigger length")
    axes[0].set_ylabel("ASR-t")
    axes[1].set_title("ASR-t vs poisoned entries")
    axes[1].set_xlabel("Poisoned entries")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend()
    savefig("figure4_sweeps")
else:
    print(f"Create {figure4_path} after sweeps finish.")

In [ ]:
# Table 2 ablation plan. The exact flags depend on the local repo version.
# Start by checking trigger_optimization.py --help, then map each loss-removal setting here.
run_command(["python", "algo/trigger_optimization.py", "--help"])

ablation_registry = pd.DataFrame([
    {"ablation": "full", "expected_change": "baseline AgentPoison", "flag_to_use": "none", "ASR-r": np.nan, "ASR-a": np.nan, "ASR-t": np.nan, "ACC": np.nan},
    {"ablation": "minus_uniqueness", "expected_change": "less isolated trigger cluster", "flag_to_use": "TBD from repo", "ASR-r": np.nan, "ASR-a": np.nan, "ASR-t": np.nan, "ACC": np.nan},
    {"ablation": "minus_compactness", "expected_change": "less compact retrieved poisons", "flag_to_use": "TBD from repo", "ASR-r": np.nan, "ASR-a": np.nan, "ASR-t": np.nan, "ACC": np.nan},
    {"ablation": "minus_target_generation", "expected_change": "lower target action/outcome", "flag_to_use": "TBD from repo", "ASR-r": np.nan, "ASR-a": np.nan, "ASR-t": np.nan, "ACC": np.nan},
    {"ablation": "minus_coherence", "expected_change": "less natural trigger", "flag_to_use": "disable --ppl_filter if equivalent", "ASR-r": np.nan, "ASR-a": np.nan, "ASR-t": np.nan, "ACC": np.nan},
])
write_table(ablation_registry, "table2_ablation_registry")
ablation_registry

In [ ]:
# Tables 4 and 6: defense registry.
# Fill with official paper values and our reproduced values after running the defense settings.
defense_registry = pd.DataFrame([
    {"agent": agent, "defense": defense, "metric": metric, "paper_value": np.nan, "ours_mean": np.nan, "notes": ""}
    for agent in ["qa", "ehr"]
    for defense in ["none", "perplexity_filtering", "query_rephrasing"]
    for metric in ["ASR-r", "ASR-a", "ASR-t", "ACC"]
])
write_table(defense_registry, "tables4_6_defense_registry")
defense_registry.head(12)

In [ ]:
# Figures 2 and 11: embedding-space visualization.
# Expected NPZ format: arrays named embeddings and labels.
def pca_2d_numpy(embeddings: np.ndarray) -> np.ndarray:
    centered = embeddings - embeddings.mean(axis=0, keepdims=True)
    _, _, vt = np.linalg.svd(centered, full_matrices=False)
    return centered @ vt[:2].T


embedding_path = TABLE_DIR / "embedding_vectors.npz"
if embedding_path.exists():
    data = np.load(embedding_path, allow_pickle=True)
    embeddings = data["embeddings"]
    labels = data["labels"]
    coords = pca_2d_numpy(embeddings)
    fig, ax = plt.subplots(figsize=(6, 5))
    for label in sorted(set(labels)):
        idx = labels == label
        ax.scatter(coords[idx, 0], coords[idx, 1], s=16, alpha=0.75, label=label)
    ax.set_title("Retriever embedding space under AgentPoison")
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.legend()
    ax.grid(alpha=0.2)
    savefig("embedding_space_agentpoison")
else:
    print(f"No embedding dump yet. Save embeddings to {embedding_path} with arrays: embeddings, labels.")

In [ ]:
# Presentation asset manifest: rerun this before making slides.
assets = []
for folder, kind in [(FIG_DIR, "figure"), (TABLE_DIR, "table"), (LOG_DIR, "log")]:
    for path in sorted(folder.glob("*")):
        assets.append({"kind": kind, "path": str(path.relative_to(PROJECT_ROOT)), "modified": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds")})

asset_manifest = pd.DataFrame(assets)
manifest_path = OUTPUT_ROOT / "presentation_asset_manifest.csv"
asset_manifest.to_csv(manifest_path, index=False)
print("saved", manifest_path)
asset_manifest

## Immediate Colab Checklist

- Select a GPU runtime before running setup cells.
- Keep `USE_GOOGLE_DRIVE = True` if you want outputs to survive Colab disconnects.
- Clone `https://github.com/BillChan226/AgentPoison.git` with the setup cell.
- Download the official unified datasets into `ReAct/database` and `EhrAgent/database`.
- Set `INSTALL_DEPS = True` only if imports fail; do not downgrade Colab's PyTorch unless necessary.
- Run one `qa` trigger optimization and one attacked/benign `qa` evaluation before launching the full grid.
- Fill `outputs/tables/table1_registry.csv` as runs complete; rerun plotting cells to save slide figures.

Keep Agent-Driver as stretch because it needs the nuScenes/motion-planner setup and is heavier than the first two agents.